# 保存本轮结果：完整 v1 归档与三组结果表

这部分接在《CLIP Student：中文分步》**第 12 节之后**，替代原第 13 节。也可以单独打开运行；不需要 GPU。

交付内容：九个最佳 checkpoint（三组 × 三个种子）、训练配置与完整历史、恢复文件、正式测试报告、逐样本预测、已有三组结果表、校准参数、特征缓存、数据清单、代码与环境记录。

三组为 CE、KD_raw、KD_calibrated；保留三份已有 CSV：`test_per_seed.csv`、`test_mean_std.csv`、`paired_deltas.csv`。不重训、不重测、不改写本轮结果；缺文件或校验不通过时明确报错。原始图片及其下载包不重复收录，恢复原始图片需使用原数据来源。

每次导出创建带北京时间与随机标识的新目录，旧目录保持不变。只有包含 `COMPLETE.json` 的交付目录才表示归档成功；中断后可直接重跑，失败目录会保留供排查。


In [ ]:
# 在 Colab 中单独打开时挂载 Drive；接在原 notebook 后也可运行。
import sys
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

# 这里必须指向“已经完成本轮训练和第 12 节汇总”的目录。
WORK_V1 = Path('/content/drive/MyDrive/CLIP_Jev_Student_v1')
ARCHIVE_ROOT = WORK_V1.parent / 'CLIP_Jev_archives'
print('本轮只读来源：', WORK_V1)
print('新建归档位置：', ARCHIVE_ROOT)


## 校验并归档

下格定义归档函数，再下一格开始运行。大体积特征缓存将完整写入 ZIP，之后逐文件校验，因此需要足够的 Drive 空间和运行时间。归档期间不要继续修改本轮训练目录。

校验九个 checkpoint 的 SHA-256、三组种子组合、各轮训练记录、配置与模型锁定信息，并核对逐种子表和报告指标。随后对包内全部文件逐一比对 SHA-256。均值标准差表和配对差值表按已有文件原样保存。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import csv
import hashlib
import json
import uuid
import zipfile


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def archive_v1(work, archive_root=None):
    """只读本轮源目录；每次调用都创建新交付目录，失败时不发布完整 ZIP。"""
    work = Path(work).resolve(strict=True)
    students, results = work / 'students', work / 'student_results'
    handoff, features = work / 'handoff', work / 'features'
    methods = ['CE', 'KD_raw', 'KD_calibrated']
    seeds = [20261010, 20261011, 20261012]
    expected = {(m, s) for m in methods for s in seeds}
    tables = ['test_per_seed.csv', 'test_mean_std.csv', 'paired_deltas.csv']

    def require(condition, message):
        if not condition:
            raise ValueError(message)

    def read_json(path):
        return json.loads(path.read_text(encoding='utf-8'))

    required = [students / n for n in ['selection.json', 'training_lock.json', 'history.jsonl']]
    required += [results / n for n in ['student_test_report.json', 'evaluation_lock.json',
                                     'calibrated_vs_raw_bootstrap.json', *tables]]
    required += [work / 'student_environment.json', work / 'images_verified.json',
                 features / 'clip_revision.json', handoff / 'results/calibration.json']
    required += [features / f'{s}.npz' for s in ['train', 'val_select', 'test']]
    required += [handoff / 'manifests' / f'{s}.jsonl' for s in ['train', 'val_fit', 'val_select', 'test']]
    required += [handoff / n for n in ['student.py', 'resumable_train.py', 'study.py', 'reliable_io.py']]
    required += [handoff / 'predictions/train/jev.jsonl']
    required += [results / f'{m}_{s}_test.jsonl' for m, s in sorted(expected)]
    missing = [str(p) for p in required if not p.is_file()]
    require(not missing, '缺少交付文件，请完成原 notebook 第 9–12 节：\n' + '\n'.join(missing))
    selection = read_json(students / 'selection.json')
    configs = selection['configs']
    require(len(configs) == 9 and {(c['method'], c['seed']) for c in configs} == expected,
            '必须包含预定的三组 × 三种子，共九个 checkpoint。')
    checkpoint_paths = set()
    for c in configs:
        p = (students / c['checkpoint']).resolve(strict=True)
        require(p.is_relative_to(students.resolve()) and p.is_file(), 'checkpoint 路径越界或不存在。')
        checkpoint_paths.add(p)
        require(sha256(p) == c['checkpoint_sha256'], f'模型校验失败：{p.name}')
    require(len(checkpoint_paths) == 9, '九次实验必须对应九个独立 checkpoint。')
    training = read_json(students / 'training_lock.json')
    history = [json.loads(line) for line in (students / 'history.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    for m, s in expected:
        epochs = [r['epoch'] for r in history if r['method'] == m and r['seed'] == s]
        require(sorted(epochs) == list(range(1, training['epochs'] + 1)), f'训练记录不完整：{m}_{s}')
    for key, path in [('calibration_sha256', handoff / 'results/calibration.json'),
                      ('teacher_train_sha256', handoff / 'predictions/train/jev.jsonl')]:
        require(selection[key] == training[key] == sha256(path), f'配置来源校验失败：{key}')
    for split in ['train', 'val_select']:
        require(selection['feature_hashes'][split] == training['feature_hashes'][split] == sha256(features / f'{split}.npz'), '训练特征发生变化。')
    for name, digest in training['code_sha256'].items():
        require(sha256(handoff / name) == digest, f'训练代码发生变化：{name}')
    lock = read_json(results / 'evaluation_lock.json')
    for key, path in [('selection_sha256', students / 'selection.json'),
                      ('training_lock_sha256', students / 'training_lock.json'),
                      ('test_manifest_sha256', handoff / 'manifests/test.jsonl'),
                      ('clip_revision_sha256', features / 'clip_revision.json')]:
        require(lock[key] == sha256(path), f'测试锁定校验失败：{key}')
    report = read_json(results / 'student_test_report.json')
    require(report['selection_sha256'] == lock['selection_sha256'], '报告与模型选择不匹配。')
    require(set(report['scores']) == {f'{m}_{s}' for m, s in expected}, '测试报告缺少预定实验。')
    with (results / 'test_per_seed.csv').open(encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    require(len(rows) == 9 and {(r['method'], int(r['seed'])) for r in rows} == expected, '逐种子结果表不完整。')
    for r in rows:
        score = report['scores'][f"{r['method']}_{r['seed']}"]
        for metric in ['accuracy', 'macro_f1']:
            require(abs(float(r[metric]) - 100 * score[metric]) < 1e-8, '结果表与测试报告不一致。')

    # 包含原始输入、源代码、日志、恢复状态、配置、特征、报告和已有结果表。
    # 图片原始压缩包和图片文件不重复收录；清单与图片核验记录保留。
    files = []
    for folder in [handoff, features, students, results]:
        for p in sorted(folder.rglob('*')):
            if p.is_file() and '__pycache__' not in p.parts and not p.name.endswith('.partial'):
                require(not p.is_symlink() and p.resolve().is_relative_to(work), f'不支持外部链接：{p}')
                files.append(p)
    files += [work / 'student_environment.json', work / 'images_verified.json']
    inventory = [{'path': p.relative_to(work).as_posix(), 'bytes': p.stat().st_size,
                  'sha256': sha256(p)} for p in files]
    root = Path(archive_root).resolve() if archive_root else work.parent / 'CLIP_Jev_archives'
    require(not root.is_relative_to(work), '归档目录必须在本轮 WORK 之外。')
    root.mkdir(parents=True, exist_ok=True)
    run_id = 'v1_' + datetime.now(timezone(timedelta(hours=8))).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:12]
    delivery = root / run_id
    delivery.mkdir(exist_ok=False)
    manifest = {'version': 'v1', 'archive_id': run_id, 'source_work': str(work),
                'methods': methods, 'seeds': seeds, 'checkpoint_count': 9,
                'excluded': ['原始图片与图片下载压缩包', '__pycache__', '*.partial'],
                'files': inventory}
    manifest_text = json.dumps(manifest, ensure_ascii=False, indent=2)
    readme = ('# CLIP-Jev Student v1\n\n冻结 CLIP + MLP；CE、KD_raw、KD_calibrated 各三个种子。\n'
              '固定测试子集，未重新训练、测试或挑选种子。指标表使用百分数，差值使用百分点。\n'
              'std 是三个训练种子的样本标准差；bootstrap 区间表示测试样本不确定性。\n'
              'student_results/ 包含已有的逐种子表、均值标准差表和配对差值表。\n'
              'MANIFEST.json 记录每个源文件的 SHA-256。原始图片不包含在本包中。\n'
              '新实验必须另建 WORK；保留本轮目录和本包。\n')
    partial = delivery / 'CLIP_Jev_Student_v1.zip.partial'
    with zipfile.ZipFile(partial, 'x', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as z:
        for p, item in zip(files, inventory):
            z.write(p, item['path'])
        z.writestr('MANIFEST.json', manifest_text)
        z.writestr('README_zh.md', readme)
    # 验证实际打包字节；源文件若在归档期间改变，也会停止发布。
    with zipfile.ZipFile(partial) as z:
        for p, item in zip(files, inventory):
            h = hashlib.sha256()
            with z.open(item['path']) as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b''):
                    h.update(chunk)
            require(h.hexdigest() == item['sha256'] == sha256(p), f"文件在归档中改变：{item['path']}")
        table_bytes = {name: z.read('student_results/' + name) for name in tables}
    for name, data in table_bytes.items():
        with (delivery / name).open('xb') as f:
            f.write(data)
    (delivery / 'MANIFEST.json').write_text(manifest_text, encoding='utf-8')
    (delivery / 'README_zh.md').write_text(readme, encoding='utf-8')
    final = delivery / 'CLIP_Jev_Student_v1.zip'
    digest = sha256(partial)
    # 专属新目录中发布；重跑创建另一个目录，绝不复用旧交付文件。
    partial.rename(final)
    (delivery / 'SHA256SUMS.txt').write_text(f'{digest}  {final.name}\n', encoding='utf-8')
    (delivery / 'COMPLETE.json').write_text(json.dumps({'status': 'complete', 'zip_sha256': digest,
        'files_verified': len(files), 'checkpoints_verified': 9}, indent=2), encoding='utf-8')
    print('v1 归档校验完成：', delivery)
    print('ZIP：', final, '\n九个 checkpoint、配置、训练记录、测试报告和已有结果表均已保存。')
    return delivery


In [ ]:
DELIVERY_V1 = archive_v1(WORK_V1, ARCHIVE_ROOT)
print('交付目录文件：')
for p in sorted(DELIVERY_V1.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1024**2:.2f} MB)')


## 查看已归档的结果表

以下只读取归档成功后导出的 CSV，不改写原结果。下载整个新交付目录，或下载 ZIP、三份 CSV 及 SHA256SUMS.txt。ZIP 内也包含三份 CSV。


In [ ]:
import pandas as pd
from IPython.display import display
for name in ['test_per_seed.csv', 'test_mean_std.csv', 'paired_deltas.csv']:
    print(name)
    kwargs = {'header': [0, 1], 'index_col': 0} if name == 'test_mean_std.csv' else {}
    display(pd.read_csv(DELIVERY_V1 / name, **kwargs))


## 后续新实验目录

保留 `CLIP_Jev_Student_v1` 与本次归档。在开始新的实验时，先把原训练 notebook 的 `WORK` 改为一个全新目录（例如 `CLIP_Jev_Student_v2_日期_实验名`），再执行初始化与训练。不要只改变种子或超参数后继续写入 v1；同一轮断线恢复才使用原目录。

本 notebook 不创建新的实验，不替你更改当前训练路径。交付表中的结果来自已完成的测试报告，不预设校准蒸馏一定优于基线。
